## ⚙️ Setup Notes

This notebook was developed on Google Colab with the following setup:
- Source data mounted from Google Drive (`/content/drive/MyDrive/medbrain-squad/`)
- Requires GPU runtime (T4 recommended)
- Update the `DATA_PATH` variable in the Config cell to point to your data location

To reproduce:
1. Upload sample data from `/data/` in the GitHub repo to your Drive
2. Mount Drive and update paths
3. Run all cells


### Install packages

In [ ]:
!pip install -U trl peft transformers accelerate bitsandbytes


### Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# Define the exact path to the file in your Drive
base_path = '/content/drive/MyDrive/Data_Science/Kaggle/MedGemma/'

## 🔑 Authentication
MedGemma is a **gated model**. Before running this notebook:
1. Accept terms at [huggingface.co/google/medgemma-1.5-4b-it](https://huggingface.co/google/medgemma-1.5-4b-it)
2. Generate a HuggingFace token at [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens)
3. Set your token below (never commit real tokens to GitHub)


In [ ]:
from google.colab import userdata
from huggingface_hub import login

# 1. Get the token from Colab's secret manager
hf_token = userdata.get('HF_TOKEN')

# 2. Log in to Hugging Face
login(token=hf_token)


### Import packages

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, DataCollatorForLanguageModeling
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig
import gc
import os
from datasets import load_dataset

### Data Collators

In [ ]:
class ManualCompletionCollator(DataCollatorForLanguageModeling):
    def __init__(self, response_template, tokenizer, *args, **kwargs):
        # We set mlm=False because this is causal language modeling
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = response_template

    def __call__(self, features, return_tensors="pt"):
        # 1. Tokenize the raw 'text' fields from the batch
        # This is the fix for the "Unable to create tensor" nesting error
        if isinstance(features[0], dict) and "text" in features[0]:
            texts = [f["text"] for f in features]
            batch = self.tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=2048
            )
            # Create labels: start with a copy of input_ids
            batch["labels"] = batch["input_ids"].clone()
        else:
            # Fallback for already tokenized features
            batch = super().__call__(features, return_tensors=return_tensors)

        # 2. Gemma 3 Requirement: token_type_ids
        if "token_type_ids" not in batch:
            batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])

        # 3. Masking Logic: Find 'model' then the first '{'
        bracket_id = self.tokenizer.encode("{", add_special_tokens=False)[-1]
        model_token_ids = self.tokenizer.encode("model", add_special_tokens=False)

        for i in range(len(batch["input_ids"])):
            input_ids = batch["input_ids"][i].tolist()
            found_model_idx = -1

            # Search for the 'model' marker
            for idx in range(len(input_ids) - len(model_token_ids)):
                if input_ids[idx : idx + len(model_token_ids)] == model_token_ids:
                    found_model_idx = idx
                    break

            # If found, find the first '{' after it and mask everything before it
            if found_model_idx != -1:
                for idx in range(found_model_idx, len(input_ids)):
                    if input_ids[idx] == bracket_id:
                        batch["labels"][i, :idx] = -100 # Mask prompt & metadata
                        break

            # Also mask padding tokens in labels
            batch["labels"][i, batch["input_ids"][i] == self.tokenizer.pad_token_id] = -100

        return batch

In [ ]:
class AuditorCoTCollator(DataCollatorForLanguageModeling):
    def __init__(self, tokenizer, *args, **kwargs):
        super().__init__(tokenizer=tokenizer, mlm=False, *args, **kwargs)
        self.response_template = "model\n"

    def __call__(self, features, return_tensors="pt"):
        if isinstance(features[0], dict) and "text" in features[0]:
            texts = [f["text"] for f in features]
            batch = self.tokenizer(
                texts, return_tensors="pt", padding=True, truncation=True, max_length=2048
            )
            batch["labels"] = batch["input_ids"].clone()
        else:
            batch = super().__call__(features, return_tensors=return_tensors)

        if "token_type_ids" not in batch:
            batch["token_type_ids"] = torch.zeros_like(batch["input_ids"])

        model_token_ids = self.tokenizer.encode(self.response_template, add_special_tokens=False)

        for i in range(len(batch["input_ids"])):
            input_ids = batch["input_ids"][i].tolist()
            found_model_idx = -1
            for idx in range(len(input_ids) - len(model_token_ids)):
                if input_ids[idx : idx + len(model_token_ids)] == model_token_ids:
                    found_model_idx = idx + len(model_token_ids)
                    break

            if found_model_idx != -1:
                batch["labels"][i, :found_model_idx] = -100 # Mask only the prompt

            batch["labels"][i, batch["input_ids"][i] == self.tokenizer.pad_token_id] = -100
        return batch

### Scriber SFT run

In [ ]:
# --- CONFIGURATION ---
MODEL_ID = "google/medgemma-1.5-4b-it"

SCRIBE_FILE_NAME = 'medgemma_scriber_sft_ready.jsonl'
AUDITOR_FILE_NAME = 'medgemma_auditor_sft_ready_clean_and_dirty.jsonl'

SCRIBE_DATA = os.path.join(base_path, SCRIBE_FILE_NAME)
AUDITOR_DATA = os.path.join(base_path, AUDITOR_FILE_NAME)

def train_adapter(dataset_path, adapter_name):
    print(f"\n🚀 KICKING OFF: {adapter_name}")

    # 1. Load Model (BF16 for A100)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        trust_remote_code=True
    )

    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    # 2. LoRA Configuration
    peft_config = LoraConfig(
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(model, peft_config)
    model.print_trainable_parameters()

    # 3. Load Dataset (Keep it as raw text for the Trainer)
    train_data = load_dataset('json', data_files=dataset_path, split='train')

    # Split if large enough
    if len(train_data) > 10:
        split_dataset = train_data.train_test_split(test_size=0.1, seed=42)
        train_data = split_dataset['train']
        print(f"✅ Split dataset. Training on {len(train_data)} rows.")

    # 4. Data Collator
    data_collator = ManualCompletionCollator(
        response_template="model\n",
        tokenizer=tokenizer
    )

    # 5. SFT Configuration
    training_args = SFTConfig(
        output_dir=f"./results_{adapter_name}",
        dataset_text_field="text",
        max_length=2048,
        packing=False,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        gradient_accumulation_steps=4,
        gradient_checkpointing=True,
        optim="paged_adamw_8bit",
        learning_rate=2e-4,
        fp16=False,
        bf16=True,
        logging_steps=5,
        save_strategy="no",
        max_grad_norm=1.0,
        warmup_ratio=0.03,
        lr_scheduler_type="cosine",
        report_to="none",
        remove_unused_columns=False, # Required so 'text' stays accessible to tokenizer
        dataset_kwargs={
            "add_special_tokens": False
        }
    )

    # 6. SFT Trainer
    trainer = SFTTrainer(
        model=model,
        train_dataset=train_data,
        args=training_args,
        processing_class=tokenizer,
        data_collator=data_collator
    )

    # 7. Execute & Save
    trainer.train()

    # --- DRIVE SAVING LOGIC ---
    drive_save_path = f"/content/drive/MyDrive/Data_Science/Kaggle/MedGemma/final_{adapter_name}"
    os.makedirs(drive_save_path, exist_ok=True)

    print(f"💾 Saving to Drive: {drive_save_path}")
    trainer.model.save_pretrained(drive_save_path)
    tokenizer.save_pretrained(drive_save_path)

    # --- MEMORY CLEANUP ---
    del trainer
    if 'model' in locals(): del model

    gc.collect()
    torch.cuda.empty_cache()
    print(f"✅ Finished: {adapter_name}")

# --- THE RUN ---
try:
    if os.path.exists(SCRIBE_DATA):
        train_adapter(SCRIBE_DATA, "scribe")

    # Create the Auditor-specific collator
    #auditor_collator = AuditorCoTCollator(tokenizer=tokenizer)

    # Re-run training JUST for the Auditor
    #try:
    #    if os.path.exists(AUDITOR_DATA):
    #        # We pass the new collator into the training function
    #        # (Make sure to update your train_adapter function signature to accept a collator if needed)
    #        print("🕵️ KICKING OFF THE PROTECTOR: AUDITOR TRAINING")
    #        train_adapter(AUDITOR_DATA, "auditor", custom_collator=auditor_collator)
    #except Exception as e:
    #    print(f"🚨 RUN FAILED: {e}")

    #    print("\n🏁 ALL DONE. SHUTTING DOWN IN 10s...")
    #    import time
    #    time.sleep(10)
    #    from google.colab import runtime
    #    runtime.unassign()

except Exception as e:
    print(f"🚨 RUN FAILED: {e}")


🚀 KICKING OFF: scribe


config.json:   0%|          | 0.00/2.55k [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json:   0%|          | 0.00/90.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

...

Tokenizing train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/450 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
5,1.538047
10,0.851827
15,0.434484
20,0.321834
25,0.275959
30,0.226901
35,0.186671
40,0.161736
45,0.144777
50,0.138842


💾 Saving to Drive: /content/drive/MyDrive/Data_Science/Kaggle/MedGemma/final_scribe
✅ Finished: scribe


### Auditor SFT run

In [ ]:
import torch
import os
import gc
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model
from trl import SFTConfig, SFTTrainer
from datasets import load_dataset

MODEL_ID = "google/medgemma-1.5-4b-it"

SCRIBE_FILE_NAME = 'medgemma_scriber_sft_ready.jsonl'
AUDITOR_FILE_NAME = 'medgemma_auditor_sft_ready_clean_and_dirty.jsonl'

SCRIBE_DATA = os.path.join(base_path, SCRIBE_FILE_NAME)
AUDITOR_DATA = os.path.join(base_path, AUDITOR_FILE_NAME)

def train_adapter(dataset_path, adapter_name):
    print(f"\n🚀 KICKING OFF: {adapter_name}")

    # 1. Load Model (BF16 for A100)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="cuda",
        trust_remote_code=True
    )

# 1. Re-initialize Tokenizer and Base Model
print("📡 Loading Base Model for Auditor Run...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="cuda",
    trust_remote_code=True
)

# 2. Apply LoRA
peft_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=["q_proj", "o_proj", "k_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)
model = get_peft_model(model, peft_config)

# 3. Setup the Auditor-Specific (CoT) Collator
# This version ensures the model learns the "Reasoning:" section!
data_collator = AuditorCoTCollator(tokenizer=tokenizer)

# 4. Load Auditor Dataset
auditor_dataset = load_dataset('json', data_files=AUDITOR_DATA, split='train')
if len(auditor_dataset) > 10:
    auditor_dataset = auditor_dataset.train_test_split(test_size=0.1, seed=42)['train']

# 5. Auditor SFT Configuration
training_args = SFTConfig(
    output_dir="./results_auditor",
    dataset_text_field="text",
    max_length=2048,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    bf16=True,
    logging_steps=5,
    save_strategy="no",
    remove_unused_columns=False,
    report_to="none"
)

# 6. Kick off Auditor Trainer
print("🕵️ TRAINING THE AUDITOR (Chain-of-Thought Mode)...")
trainer = SFTTrainer(
    model=model,
    train_dataset=auditor_dataset,
    args=training_args,
    processing_class=tokenizer,
    data_collator=data_collator
)

trainer.train()

# 7. Final Save to Drive
drive_save_path = f"/content/drive/MyDrive/Data_Science/Kaggle/MedGemma/final_auditor"
os.makedirs(drive_save_path, exist_ok=True)
trainer.model.save_pretrained(drive_save_path)
tokenizer.save_pretrained(drive_save_path)

print(f"✅ Auditor successfully banked at: {drive_save_path}")

📡 Loading Base Model for Auditor Run...


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Generating train split: 0 examples [00:00, ? examples/s]

🕵️ TRAINING THE AUDITOR (Chain-of-Thought Mode)...


Adding EOS to train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/900 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'bos_token_id': 2, 'pad_token_id': 1}.


Step,Training Loss
5,0.794280
10,0.569347
15,0.528527
20,0.438217
25,0.344355
30,0.246306
35,0.198779
40,0.166188
45,0.127628
50,0.128054


✅ Auditor successfully banked at: /content/drive/MyDrive/Data_Science/Kaggle/MedGemma/final_auditor
